In [18]:
from dgp.datasets import SynchronizedSceneDataset
import os
import imageio
import numpy as np
from tqdm import tqdm

# 1) 저장할 카메라 목록 (JSON의 datum_name과 동일하게 대문자)
camera_list = [
    'CAMERA_01', 'CAMERA_05', 'CAMERA_06',
    'CAMERA_07', 'CAMERA_08', 'CAMERA_09'
]

output_base = 'output'
# 2) 출력 디렉터리 생성
for cam in camera_list:
    os.makedirs(os.path.join(output_base, 'rgb', cam), exist_ok=True)
    os.makedirs(os.path.join(output_base, 'depth_gt_sparse', cam), exist_ok=True)

# 3) 데이터셋 초기화
dataset = SynchronizedSceneDataset(
    'ddad.json',        # ← 실제 경로로 수정
    datum_names=('lidar',) + tuple(cam.lower() for cam in camera_list),
    generate_depth_from_datum='lidar',
    split='train'
)

# 4) 순회하며 저장
for idx, sample in enumerate(tqdm(dataset, desc='Processing samples')):
    inner = sample[0]  # 리스트 안의 OrderedDict 리스트

    # 4-1) 각 카메라 항목에서 depth와 rgb를 꺼내 저장
    for entry in inner:
        name = entry['datum_name']
        if name == 'LIDAR':
            continue

        # 4-1-1) RGB 저장
        rgb_img = entry['rgb']    # PIL.Image
        rgb_path = os.path.join(
            output_base, 'rgb', name, f'{idx:06d}.png'
        )
        rgb_img.save(rgb_path)

        # 4-1-2) depth 저장
        depth_m = entry['depth']  # (H, W), meters
        depth_mm = (depth_m * 1000.0).astype(np.uint16)
        depth_path = os.path.join(
            output_base, 'depth_gt_sparse', name, f'{idx:06d}.png'
        )
        imageio.imwrite(depth_path, depth_mm)

print("완료: 모든 카메라의 RGB 및 sparse depth 저장되었습니다.")


Processing samples: 100%|██████████| 12650/12650 [11:39:09<00:00,  3.32s/it]  

완료: 모든 카메라의 RGB 및 sparse depth 저장되었습니다.
